In [1]:
import os

# Option A – hard-code (only for local use, never commit to git)
# os.environ["GROQ_API_KEY"] = "gsk_..."

# Option B – prompt at runtime
if not os.environ.get("GROQ_API_KEY"):
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

print("✅ API key configured.")

✅ API key configured.


In [2]:
import urllib.request, pathlib

DATASETS = {
    "regression-dataset.csv":     "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/N0CceRlquaf9q85PK759WQ/regression-dataset.csv",
    "classification-dataset.csv": "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7J73m6Nsz-vmojwab91gMA/classification-dataset.csv",
}

for fname, url in DATASETS.items():
    if not pathlib.Path(fname).exists():
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, fname)
    else:
        print(f"✓ {fname} already present")

print("\nDatasets ready.")


Datasets ready.


In [4]:
%pip install -q langchain langchain-groq groq pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import glob
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_groq import ChatGroq

print("✅ All imports successful.")

✅ All imports successful.


In [8]:
# Shared cache — lives at module scope so every tool sees the same dict
DATAFRAME_CACHE: Dict[str, pd.DataFrame] = {}


def _load(path: str) -> pd.DataFrame:
    """Load a CSV into DATAFRAME_CACHE if not already there, then return it."""
    if path not in DATAFRAME_CACHE:
        DATAFRAME_CACHE[path] = pd.read_csv(path)
    return DATAFRAME_CACHE[path]

In [9]:
@tool
def list_csv_files() -> Optional[List[str]]:
    """List every CSV file name found in the current working directory.

    Returns:
        A list of CSV file names, or None if no CSV files exist.
    """
    matches = glob.glob(os.path.join(os.getcwd(), "*.csv"))
    return [os.path.basename(p) for p in matches] or None


# Quick sanity check
print("Available CSVs:", list_csv_files.invoke({}))

Available CSVs: ['classification-dataset.csv', 'regression-dataset.csv']


In [10]:
@tool
def preload_datasets(paths: List[str]) -> str:
    """Load one or more CSV files into the in-memory cache.

    Call this before any tool that needs to read a dataset. Files already in
    the cache are skipped, so it is safe to call repeatedly.

    Args:
        paths: File names or paths of the CSV files to load.

    Returns:
        A summary of which files were loaded vs. already cached.
    """
    loaded, already_cached = [], []
    for p in paths:
        if p in DATAFRAME_CACHE:
            already_cached.append(p)
        else:
            try:
                _load(p)
                loaded.append(p)
            except FileNotFoundError:
                return f"Error: '{p}' not found. Check the file name."
            except Exception as exc:
                return f"Error loading '{p}': {exc}"

    return f"Loaded: {loaded}  |  Already cached: {already_cached}"

In [11]:
@tool
def get_dataset_summaries(dataset_paths: List[str]) -> List[Dict[str, Any]]:
    """Return schema metadata for one or more datasets.

    Provides column names, data types, row / column counts, and the number
    of unique values per column — enough for the agent to decide whether
    a column is categorical (classification target) or continuous (regression
    target) without loading the full data into the context window.

    Args:
        dataset_paths: File names or paths of the CSV files.

    Returns:
        A list of summary dicts, one per dataset.
    """
    summaries = []
    for path in dataset_paths:
        try:
            df = _load(path)
        except Exception as exc:
            summaries.append({"file_name": path, "error": str(exc)})
            continue

        summaries.append({
            "file_name":    path,
            "rows":         len(df),
            "columns":      len(df.columns),
            "column_names": df.columns.tolist(),
            "data_types":   df.dtypes.astype(str).to_dict(),
            "unique_counts": df.nunique().to_dict(),   # helps detect classification targets
            "missing_values": df.isnull().sum().to_dict(),
        })
    return summaries

In [12]:
# Safe allow-list prevents the agent from calling destructive methods
_ALLOWED_METHODS = {"head", "tail", "describe", "info", "shape", "dtypes", "columns", "sample"}


@tool
def call_dataframe_method(file_name: str, method: str) -> str:
    """Run a read-only pandas DataFrame method and return the result as text.

    Allowed methods: head, tail, describe, info, shape, dtypes, columns, sample.

    Args:
        file_name: The CSV file name (must already be in the cache or on disk).
        method:    The pandas method to call (no arguments).

    Returns:
        String representation of the method output, or an error message.
    """
    if method not in _ALLOWED_METHODS:
        return f"Method '{method}' is not allowed. Choose from: {sorted(_ALLOWED_METHODS)}."

    try:
        df = _load(file_name)
    except FileNotFoundError:
        return f"File '{file_name}' not found in cache or on disk."
    except Exception as exc:
        return f"Error loading '{file_name}': {exc}"

    func = getattr(df, method, None)
    if func is None:
        return f"'{method}' is not a valid DataFrame attribute."

    try:
        result = func() if callable(func) else func
        return str(result)
    except Exception as exc:
        return f"Error calling '{method}': {exc}"

In [13]:
@tool
def evaluate_classification_dataset(file_name: str, target_column: str) -> Dict[str, Any]:
    """Train a Random Forest classifier and report accuracy on a held-out test set.

    The dataset is split 80/20 (train/test) with a fixed random seed for
    reproducibility. Use this tool when the target column contains discrete
    class labels.

    Args:
        file_name:     CSV file name containing the dataset.
        target_column: Name of the column to predict.

    Returns:
        Dict with 'accuracy' (0–1) and 'n_classes', or an 'error' key on failure.
    """
    try:
        df = _load(file_name)
    except Exception as exc:
        return {"error": str(exc)}

    if target_column not in df.columns:
        return {"error": f"Column '{target_column}' not found. Available: {df.columns.tolist()}"}

    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Drop non-numeric features silently (keep it simple)
    X = X.select_dtypes(include=[np.number])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    return {
        "accuracy":  round(acc, 4),
        "n_classes": int(y.nunique()),
    }

In [14]:
@tool
def evaluate_regression_dataset(file_name: str, target_column: str) -> Dict[str, Any]:
    """Train a Random Forest regressor and report R² and RMSE on a held-out test set.

    Use this tool when the target column is a continuous numeric variable.

    Args:
        file_name:     CSV file name containing the dataset.
        target_column: Name of the column to predict.

    Returns:
        Dict with 'r2_score' and 'rmse', or an 'error' key on failure.
    """
    try:
        df = _load(file_name)
    except Exception as exc:
        return {"error": str(exc)}

    if target_column not in df.columns:
        return {"error": f"Column '{target_column}' not found. Available: {df.columns.tolist()}"}

    X = df.drop(columns=[target_column]).select_dtypes(include=[np.number])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return {
        "r2_score": round(r2_score(y_test, preds), 4),
        "rmse":     round(float(np.sqrt(mean_squared_error(y_test, preds))), 4),
    }

In [15]:
# All tools in one place — easy to extend
TOOLS = [
    list_csv_files,
    preload_datasets,
    get_dataset_summaries,
    call_dataframe_method,
    evaluate_classification_dataset,
    evaluate_regression_dataset,
]

# ── LLM ──────────────────────────────────────────────────────────────────────
# llama-3.3-70b-versatile is Groq's best general-purpose model for tool use.
# Switch to llama-3.1-8b-instant for faster / cheaper responses.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,          # deterministic tool selection
    max_retries=2,
)

# ── Prompt ───────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are DataWizard, an expert data science assistant.

Your workflow for any dataset question:
1. Call list_csv_files to discover available data.
2. Call preload_datasets to load the relevant file(s).
3. Call get_dataset_summaries to understand structure.
4. Decide whether the task is classification or regression by inspecting
   the target column's unique value count and data type.
5. Call the appropriate evaluation tool and report the result clearly.

Always give a concise, plain-English explanation alongside any numbers.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system",  SYSTEM_PROMPT),
    ("human",   "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# ── Agent + Executor ──────────────────────────────────────────────────────────
agent = create_tool_calling_agent(llm, TOOLS, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,               # prints each ReAct step
    handle_parsing_errors=True, # gracefully recover from malformed LLM output
    max_iterations=10,          # prevent infinite loops on edge cases
    return_intermediate_steps=False,
)

print("✅ Agent ready.")

✅ Agent ready.


In [16]:
DEMO_QUERIES = [
    "What CSV datasets are available?",
    "Give me a summary of all available datasets, including their shapes and column types.",
    "Evaluate the machine learning performance of every dataset and tell me the results in plain English.",
]

for i, query in enumerate(DEMO_QUERIES, 1):
    print(f"\n{'='*70}")
    print(f"DEMO QUERY {i}: {query}")
    print("="*70)
    result = agent_executor.invoke({"input": query})
    print(f"\n🤖 DataWizard:\n{result['output']}")


DEMO QUERY 1: What CSV datasets are available?


> Entering new AgentExecutor chain...

Invoking: `list_csv_files` with `{}`


['classification-dataset.csv', 'regression-dataset.csv']
Invoking: `preload_datasets` with `{'paths': ['classification-dataset.csv', 'regression-dataset.csv']}`


Loaded: ['classification-dataset.csv', 'regression-dataset.csv']  |  Already cached: []
Invoking: `get_dataset_summaries` with `{'dataset_paths': ['classification-dataset.csv', 'regression-dataset.csv']}`


[{'file_name': 'classification-dataset.csv', 'rows': 569, 'columns': 31, 'column_names': ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture',